In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import matplotlib.animation as animation
from IPython.display import HTML, display

Pobranie najbliższych sąsiadów

In [ ]:
def get_direct_neighbors(grid, x, y):
    neighbors = []
    grid_size = grid.shape[0]
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0:
                continue
            nx, ny = x + dx, y + dy
            if 0 <= nx < grid_size and 0 <= ny < grid_size:
                neighbors.append(grid[nx, ny])
    return neighbors

Sprawdzenie czy agent jest usatysfakcjonowany

In [ ]:
def is_satisfied(grid, x, y, tolerance):
    agent = grid[x, y]
    if agent == 0:
        return True

    neighbors = get_direct_neighbors(grid, x, y)
    if not neighbors:
        return False

    same_type = sum(1 for n in neighbors if n == agent)
    return same_type / len(neighbors) >= tolerance

Inicjalizacja Siatki

In [ ]:
def initialize_grid(grid_size=50, density=0.75):
    num_cells = grid_size**2
    num_agents = int(num_cells * density)
    num_agents = (num_agents // 2) * 2  # Niech liszba agentów będzie parzysta

    # Podzielenie agentów
    agents = np.zeros(num_cells, dtype=np.int8)
    agents[:num_agents//2] = 1
    agents[num_agents//2:num_agents] = 2

    # Rozrzucenie agentów
    np.random.shuffle(agents)
    grid = agents.reshape(grid_size, grid_size)

    # Ustawienie pustych miejsc
    empty_spots = set((x, y) for x in range(grid_size) for y in range(grid_size) if grid[x, y] == 0)

    return grid, empty_spots

Pobranie losowego Agenta

In [ ]:
def getRandomAgent(grid, unhappy_agents):
    if not unhappy_agents:
        return None
    return list(unhappy_agents)[np.random.randint(0, len(unhappy_agents))]

Jeden krok

In [ ]:
def step(agent, grid, unTolerance, empty_spots, unhappy_agents):
    if agent is None:
        return grid, empty_spots, unhappy_agents, False

    x, y = agent
    grid_size = grid.shape[0]

    # Usunięcie agenta z listy smutnych jeśli teraz zadowolony
    if is_satisfied(grid, x, y, unTolerance) or grid[x, y] == 0:
        unhappy_agents.remove(agent)
        return grid, empty_spots, unhappy_agents, False

    if empty_spots:
        # Losowy punkt z pustych
        new_x, new_y = list(empty_spots)[np.random.randint(0, len(empty_spots))]

        # Ruszenie agenta
        agent_type = grid[x, y]
        grid[new_x, new_y] = agent_type
        grid[x, y] = 0

        # Aktualizacja pustych miejsc
        empty_spots.remove((new_x, new_y))
        empty_spots.add((x, y))

        # wywalenie z listy smutnych
        unhappy_agents.remove((x, y))

        # Sprawdzenie czy jest szczęśliwy w nowym miejscu
        if not is_satisfied(grid, new_x, new_y, unTolerance):
            unhappy_agents.add((new_x, new_y))

        # Sprawdzenie przeszłych i nowych sąsiadów
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                if dx == 0 and dy == 0:
                    continue
                for (nx, ny) in [(x + dx, y + dy), (new_x + dx, new_y + dy)]:
                    if 0 <= nx < grid_size and 0 <= ny < grid_size and grid[nx, ny] != 0:
                        if is_satisfied(grid, nx, ny, unTolerance):
                            unhappy_agents.discard((nx, ny))
                        else:
                            unhappy_agents.add((nx, ny))

        return grid, empty_spots, unhappy_agents, True

    return grid, empty_spots, unhappy_agents, False

Rysowanie siatki

In [ ]:
def plot_grid(grid, iteration, cmap):
    plt.figure(figsize=(6, 6))
    plt.imshow(grid, cmap=cmap, vmin=0, vmax=2)
    plt.title(f'Iteration: {iteration}')
    plt.show(block=False)
    plt.pause(0.1)
    print(f"Created plot for iteration {iteration}")

Symulacja

In [ ]:
def run_schelling_simulation(grid_size=50, density=0.75, intolerance=0.3, max_iterations=10000, plot=True, animate=False):

    grid, empty_spots = initialize_grid(grid_size, density)
    cmap = plt.cm.colors.ListedColormap(['white', 'green', 'blue'])

    unhappy_agents = set()
    for x in range(grid_size):
        for y in range(grid_size):
            if grid[x, y] != 0 and not is_satisfied(grid, x, y, intolerance):
                unhappy_agents.add((x, y))

    frames = []

    if plot and not animate:
        plot_grid(grid, 0, cmap)

    iteration = 0
    unhappy_agents_history = []

    while iteration < max_iterations:
        agent = getRandomAgent(grid, unhappy_agents)
        if agent is None:
            break

        grid, empty_spots, unhappy_agents, has_moved = step(agent, grid, intolerance, empty_spots, unhappy_agents)

        if has_moved:
            iteration += 1
            if animate:
                frames.append(grid.copy())
            elif plot and iteration in [1, 1000]:
                plot_grid(grid, iteration, cmap)

        unhappy_agents_history.append(len(list(unhappy_agents)))

    if plot and not animate:
        if iteration != 1000:
            plot_grid(grid, iteration, cmap)

    if animate:
        fig, ax = plt.subplots(figsize=(6, 6))

        def update(frame):
            ax.clear()
            ax.imshow(frame, cmap=cmap, vmin=0, vmax=2)
            ax.axis('off')
            return ax

        ani = animation.FuncAnimation(fig, update, frames=frames, interval=100, blit=False)

        try:
            get_ipython
            display(HTML(ani.to_jshtml()))
        except NameError:
            print("Running outside Jupyter, saving animation...")
            ani.save('schelling_simulation.gif', writer='imagemagick')

    if not unhappy_agents:
        print(f"All agents are satisfied - equilibrium reached at iteration {iteration}")
    elif iteration >= max_iterations:
        print(f"Reached maximum iteration limit: {max_iterations}")

    return unhappy_agents_history, grid

Wstępne symulacje i wnioski:

Dla 75%:
Punkt krytyczny: Tolerancja 26%

In [ ]:
run_schelling_simulation(grid_size=50, density=0.75, tolerance=0.25, max_iterations=10000, plot=True, animate=True)
run_schelling_simulation(grid_size=50, density=0.75, tolerance=0.26, max_iterations=10000, plot=True, animate=True)

Dla 85%:
Punkt krytyczny: Tolerancja 26%

In [ ]:
run_schelling_simulation(grid_size=50, density=0.85, tolerance=0.25, max_iterations=10000, plot=True)
run_schelling_simulation(grid_size=50, density=0.85, tolerance=0.26, max_iterations=10000, plot=True)

Dla 95%:
Punkt krytyczny: Tolerancja 26%

In [ ]:
run_schelling_simulation(grid_size=50, density=0.85, tolerance=0.25, max_iterations=10000, plot=True)
run_schelling_simulation(grid_size=50, density=0.85, tolerance=0.26, max_iterations=10000, plot=True)

Dalsza część zadania

In [ ]:
data75, grid75 = run_schelling_simulation(grid_size=50, density=0.75, tolerance=0.3, plot=False, animate=False)
data85, grid85 = run_schelling_simulation(grid_size=50, density=0.85, tolerance=0.3, plot=False, animate=False)
data95, grid95 = run_schelling_simulation(grid_size=50, density=0.95, tolerance=0.3, plot=False, animate=False)

Wykresy liczby nieszczęśliwych agentów w zależności od tolerancji

In [ ]:
plt.plot(data75)
plt.xlabel('Iteracja')
plt.ylabel('Liczba nieszczęśliwych agentów')
plt.title('Liczba nieszczęśliwych agentów od iteracji dla gęstości 75%')
plt.show()

plt.plot(data85)
plt.xlabel('Iteracja')
plt.ylabel('Liczba nieszczęśliwych agentów')
plt.title('Liczba nieszczęśliwych agentów od iteracji dla gęstości 85%')
plt.show()

plt.plot(data95)
plt.xlabel('Iteracja')
plt.ylabel('Liczba nieszczęśliwych agentów')
plt.title('Liczba nieszczęśliwych agentów od iteracji dla gęstości 95%')
plt.show()

Wykres tolerancji - Oblicza/Tworzy wykres dla symulacji puszczonej jeden raz na krok tolerancji (UWAGA: długo mieli 1m 17s)

In [ ]:
def wykres_tolerancji(step, density, grid_size=20):
    unsatf_list = []
    xi = []
    for i in range(1, int(1/step + 1)):
        #tutaj biore for'a od 1 bo tolerancja dla 0 to jakies dziwne rzeczy wychodza, do sprawdzenia czemu tak
        unsatf_agents, grid = run_schelling_simulation(grid_size=grid_size, density=density, tolerance=i*step, plot=False)
        unsatf_list.append(unsatf_agents[-1])
        xi.append(i*step)

    return unsatf_list, xi

In [ ]:
wykres_tolerancji(0.01, 0.75, grid_size=40)
wykres_tolerancji(0.01, 0.85, grid_size=40)
wykres_tolerancji(0.01, 0.95, grid_size=40)

Średnia tolerancji :
    step: okresla co jaki krok zmienia sie tolerancja
    size: z ilu symulacji liczymy srednia dla jednego kroku

In [ ]:
def srednia_tolerancji(step, density, grid_size=20, size=5):
    lista_xi = []
    all_unsatf_list = []
    print(f'Spodziewany czas to {size} * czas_jednej_symluacji')
    for i in range(size + 1):
        print(i)
        unsatf_list, xi_lisy = wykres_tolerancji(step, density, grid_size)
        all_unsatf_list.append(unsatf_list)
        lista_xi.append(xi_lisy)

    return all_unsatf_list, lista_xi

Wykresy niezadowolonych agentów w stanie końcowym w zależności od tolerancji dla gęstości

In [ ]:
density = 0.75
grid_size = 50
step_size = 0.01
number_average = 10

unsatf, lista_xi = srednia_tolerancji(step_size, density, grid_size, number_average)
mean_unsatf = np.mean(unsatf, axis=0)

plt.plot(lista_xi[0], mean_unsatf, label="Satf")
plt.title(f"Wykres Zadowolonych agentow w stanie koncowym w zal. od tol. dla gestosci {density}")
plt.legend()
plt.show()

density = 0.85

unsatf, lista_xi = srednia_tolerancji(step_size, density, grid_size, number_average)
mean_unsatf = np.mean(unsatf, axis=0)

plt.plot(lista_xi[0], mean_unsatf, label="Satf")
plt.title(f"Wykres Zadowolonych agentow w stanie koncowym w zal. od tol. dla gestosci {density}")
plt.legend()
plt.show()

density = 0.95

unsatf, lista_xi = srednia_tolerancji(step_size, density, grid_size, number_average)
mean_unsatf = np.mean(unsatf, axis=0)

plt.plot(lista_xi[0], mean_unsatf, label="Satf")
plt.title(f"Wykres Zadowolonych agentow w stanie koncowym w zal. od tol. dla gestosci {density}")
plt.legend()
plt.show()

Algorytm ktory laczy dwa elementy jezeli powinny byc w tej samej grupie w naszym przypadku kolorze:
funkcja find sprawdza jaki jest "parent", czyli reprezentat grupy,  funkcja union po prostu scala dwie komorki jezeli nie maja tego samego reprezentanta w zaleznosci od rangi jeden dolaczany jest do drugiego lub na odwrot


In [ ]:
class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))
        self.rank = [0] * size

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        rx = self.find(x)
        ry = self.find(y)
        if rx != ry:
            if self.rank[rx] < self.rank[ry]:
                self.parent[rx] = ry
            elif self.rank[rx] > self.rank[ry]:
                self.parent[ry] = rx
            else:
                self.parent[ry] = rx
                self.rank[rx] += 1

Algorytm do sprawdzania sasiadow kazdej komorki

In [ ]:
def hoshen_kopelman(grid, val):
    # tworzymy siatke wypelniona zerami

    rows = len(grid)
    cols = len(grid[0]) if rows > 0 else 0

    labels = [[0]*cols for _ in range(rows)]

    #tworzymy ta klase ktora potem wykorzystamy do laczenia grup agentow

    uf = UnionFind(rows * cols + 1)  # +1, by mieć 'bezpieczne' indeksy od 1 w górę
    current_label = 0

    # Pomocnicza funkcja do przekształcania (i, j) w indeks union-find
    def idx(i, j):
        return i * cols + j + 1  # +1 żeby 0 mieć "zarezerwowane" np. dla braku

    # przechodzimy po kazdej komorce i sprawdzamy sasiadow gornego i lewego, jezeli obaj maja 1 to
    for i in range(rows):
        for j in range(cols):

            if grid[i][j] == val:
                up_label = labels[i-1][j] if i > 0 else 0
                left_label = labels[i][j-1] if j > 0 else 0

                if up_label == 0 and left_label == 0:
                    # Nowy klaster
                    current_label += 1
                    labels[i][j] = current_label

                elif up_label != 0 and left_label == 0:
                    labels[i][j] = up_label

                elif up_label == 0 and left_label != 0:
                    labels[i][j] = left_label

                else:
                    # Oba są != 0 to bierzemy mniejszy i scalamy komorki
                    chosen_label = min(up_label, left_label)
                    labels[i][j] = chosen_label
                    uf.union(up_label, left_label)
            else:
                labels[i][j] = 0

    for i in range(rows):
        for j in range(cols):
            if labels[i][j] != 0:
                # reprezentant w union-find
                root_label = uf.find(labels[i][j])
                # Nadpisz w tablicy
                labels[i][j] = root_label

    return labels

Kopelman dla 2 grup

In [ ]:
def hoshen_kopelman_2groups(grid):
    labels_plus = hoshen_kopelman(grid, 1)
    labels_minus = hoshen_kopelman(grid, 2)
    return labels_plus, labels_minus

Funkcja wyliczająca wzór z pracy $s = \frac{2}{N_{agents}^2} \sum_{c}^{} n_c^{2}$

In [ ]:
def calculateS(label1, label2):
    cluster_sizes1 = Counter()
    rows1 = len(label1)
    cols1 = len(label1[0]) if rows1 > 0 else 0
    for i in range(rows1):
        for j in range(cols1):
            c1 = label1[i][j]
            if c1 != 0:
                cluster_sizes1[c1] += 1

    cluster_sizes2 = Counter()
    rows2 = len(label2)
    cols2 = len(label2[0]) if rows2 > 0 else 0
    for i in range(rows2):
        for j in range(cols2):
            c2 = label2[i][j]
            if c2 != 0:
                cluster_sizes2[c2] += 1

    N_agents1 = sum(cluster_sizes1.values())
    N_agents2 = sum(cluster_sizes2.values())

    N_tot = N_agents1 + N_agents2

    sum_sq1 = sum((sz * sz) for sz in cluster_sizes1.values())
    sum_sq2 = sum((sz * sz) for sz in cluster_sizes2.values())

    sq_tot = sum_sq1 + sum_sq2

    s_value = 2.0 * sq_tot / (N_tot * N_tot)
    return s_value

Symulacja do wyliczania S

In [ ]:
def simulation_of_S(grid_size=20, density=0.95, step_size=0.1):

    s_list = []
    tol_list = []

    for i in range(1, int(1/step_size + 1)):

        print(i*step_size)
        tol_list.append(i*step_size)

        unsatfTEMP, grid = run_schelling_simulation(grid_size=grid_size, density=density, tolerance=i*step_size, plot=False)
        lp, lm = hoshen_kopelman_2groups(grid)
        s_list.append(calculateS(lp, lm))

    return s_list, tol_list

Wyznaczenie S dla różnych gęstości

In [ ]:
density_s = 0.75
grid_size_s = 40
step_size_s = 0.01

s, tol = simulation_of_S(grid_size=grid_size_s, density=density_s, step_size=step_size_s)
plt.plot(tol, s, label="S")
plt.legend()
plt.title(f'S dla gestosci {density_s}, grid_size {grid_size_s}, step_size {step_size_s}')
plt.show()

density_s = 0.85

s, tol = simulation_of_S(grid_size=grid_size_s, density=density_s, step_size=step_size_s)
plt.plot(tol, s, label="S")
plt.legend()
plt.title(f'S dla gestosci {density_s}, grid_size {grid_size_s}, step_size {step_size_s}')
plt.show()

density_s = 0.95

s, tol = simulation_of_S(grid_size=grid_size_s, density=density_s, step_size=step_size_s)
plt.plot(tol, s, label="S")
plt.legend()
plt.title(f'S dla gestosci {density_s}, grid_size {grid_size_s}, step_size {step_size_s}')
plt.show()